# 11 · Turn a failure into a learning signal

**Relay failure → Gym verifier → fresh attempts → RL update → replay**

We inspect notebook 09’s injected failure and train a tiny policy to choose better tool plans. This is a real CPU optimization exercise with **12 plan-selection parameters**, not Lightning fine-tuning.

**Who does what:** Nano coordinates the lesson tools. Relay supplies failure evidence, Gym scores fresh plans, and NeMo RL loss functions update the tiny policy. Lightning and its adapter remain unchanged.

**CPU training is for demonstration purposes only.** The executable exercise updates a 12-parameter plan-selection policy. It does not train Lightning or improve its deployed weights. For actual Lightning supervised fine-tuning, follow [train_lightning.py](scripts/train_lightning.py) and [notebook 04](04-train-adapter.ipynb). The optional full Gym/RL exercise below explains the next stage.


###  verification, rewards, and policy updates

Gym **0.6.0** [environment tutorials](https://docs.nvidia.com/nemo/gym/v0.6.0/environment-tutorials/), [NeMo RL/Gym integration](https://docs.nvidia.com/nemo/rl/nightly/design-docs/nemo-gym-integration.html), and our [CPU experiment guide](../run/workshop/gym/README.md). RL's nightly documentation describes the full stack; this lesson uses only the functions pinned in [vendor/manifest.json](../run/workshop/gym/vendor/manifest.json).

retain 09's successful run and injected failure, set up the separate Gym environment, then run trace → case → learn. The cells below check the failure evidence, define `verify_plan`, implement the Gym verifier and `train_policy`, and launch those exact definitions in the isolated Gym interpreter. The fixed agent recipe continues to use [learning.py](agent/learning.py).

**What changes:** 12 logits choose among four fixed plans for three intents. Gym evaluates fresh attempts through an in-process HTTP client; PyTorch applies 40 updates using selected upstream RL functions. The failure is manually mapped to an existing verifier rule, not ingested as training tensors. Lightning, OpenClaw, and the agent loop from 09 are not retrained or redeployed. Evaluation reuses this tiny environment.

The agent uses a workshop-authored training skill. See [Skills and training](README.md#skills-and-training) for the distinction between these instructions, NeMo libraries, and the separate NeMo Skills pipeline.

**NVIDIA verifier implementation reference:** [Gym 0.6.0 single-step environment](https://docs.nvidia.com/nemo/gym/v0.6.0/environment-tutorials/single-step-environment) documents `SimpleResourcesServer`, `REVERIFY_MODE`, and `BaseVerifyResponse`. Our verifier follows those interfaces. The [RL source manifest](../run/workshop/gym/vendor/manifest.json) pins the extracted functions and file hashes; the full-stack nightly guide is background, not the recipe executed here. The training loop performs one update per fresh group, so PPO clipping is inactive in this exercise.


**Manual experiment versus agent reference:** the executable cells below are yours to edit. The OpenClaw section at the end runs the fixed workshop recipe in `agent/lessons/`; it does not ingest your edited cells or reuse manual artifacts. Setup, displays, artifact bookkeeping, and terminal/process transport remain helpers.

### Start here
Run the executable cells in order. Edit the visible requests, rules, and experiment parameters to change your manual run. The artifact helper only copies prerequisites and records output paths. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import table, bars, loss_chart, show_skill_library
from course.agent.notebook_support import NotebookRun
from course.common import chat, request, rows, BASE_URL
import os, time

lesson = NotebookRun('11-trace-and-improve')
OUTPUT = lesson.work
LIVE = True


### 1 · Read the saved trace

This cell loads the manual run from notebook 09. Its outcome may be COMPLETE or HOLD. Keep observed failures separate from our intentionally injected test.

In [ ]:
trace_path = OUTPUT/'agent-run.json'
if trace_path.exists():
    record = json.loads(trace_path.read_text())
    print('This is a saved run:',trace_path)
    print('Skill hash:',record['skill_sha256'],'Outcome:',record['status'])
    for event in record['events']:
        print(event['step'],event.get('action'), 'seconds=',event['seconds'],
              'error=',event.get('error'))
else:
    record = None
    print('No saved run yet. Execute notebook 09 to produce one; no substitute trace is used.')

In [ ]:
def check_trace(record):
    successful = [e for e in record['events'] if 'result' in e]
    tools = [e['action']['tool'] for e in successful]
    failures = []
    if record.get('released'): failures.append('released output')
    if 'triage' in tools and ('perceive' not in tools or tools.index('perceive') > tools.index('triage')):
        failures.append('triage without evidence')
    if 'draft' in tools:
        if 'triage' not in tools or tools.index('triage') > tools.index('draft'):
            failures.append('draft before triage')
        if not record['state']['triage']['gate']['valid']:
            failures.append('draft after invalid triage')
        if record['state']['draft']['source_id'] != record['state']['source']['id']:
            failures.append('source changed')
    if record['status'] not in {'COMPLETE','HOLD'}: failures.append('incomplete workflow')
    return failures
if record is not None:
    print('Contract failures:',check_trace(record))

In [ ]:
record = record
table(["Step", "Action", "Error"], [(e["step"], (e.get("action") or {}).get("tool"), e.get("error","")) for e in record["events"]])
print("Contract failures:", check_trace(record))

### 2 · Define a regression case

Deliberately inject **UNKNOWN sensor quality + CONFIRMED from one source**. The expected behavior is HOLD without a draft. The final check executes this regression separately from the missing-evidence learning exercise.

In [ ]:
case = {
    'name':'invalid-specialist-must-hold',
    'input':'one-source observation with UNKNOWN sensor quality',
    'injected_specialist':{'activity_type':'civil_gathering','sensor_quality':'UNKNOWN',
                          'confidence':'CONFIRMED','routing':'CIMIC'},
    'expected':'hold without drafting',
    'intervention':'verify contract and tool ordering before changing model weights',
}
(OUTPUT/'my-regression-case.json').write_text(json.dumps(case,indent=2))
print(json.dumps(case,indent=2))

In [ ]:
display(JSON(case, expanded=True))

### 3 · Run Gym and RL

| Plan | Reward rule |
|---|---|
| triage → draft | 0: evidence missing |
| perceive → triage → draft → review | 1: valid plan |

The old trace points to an existing Gym rule. It is not ingested as training tensors. The tool samples fresh plans and runs **40 CPU optimizer updates** using NeMo RL loss functions.

In [ ]:
import hashlib
failure_record = lesson.read('failed-agent-run.json')
failure_events = lesson.read('failed-relay-events.json')
assert failure_record['test_type'] == 'injected missing-evidence action'
assert failure_record['events'][0]['action']['tool'] == 'triage'
assert 'observation' not in failure_record['state']
failed_call = next(e for e in failure_events if e.get('name') == 'triage'
                   and e.get('scope_category') == 'end'
                   and (e.get('metadata') or {}).get('otel.status_code') == 'ERROR')
gym_output = OUTPUT / 'gym-rl'
gym_output.mkdir(exist_ok=True)
(gym_output / 'failure-case.json').write_text(json.dumps({
    'relay_event_id': failed_call['uuid'], 'plan_id': 'skip_evidence',
    'source_trace_sha256': hashlib.sha256((OUTPUT / 'failed-agent-run.json').read_bytes()).hexdigest(),
    'mapping': 'Manual mapping to a verifier rule; the trace is not training tensors.'}, indent=2))

### Define the policy actions and reward

Edit the verifier below to experiment with rewards. It executes offline state transitions; it does not call production tools.

In [ ]:
def plan_library():
    return [
        {'id': 'observe', 'tools': ['perceive', 'finish']},
        {'id': 'classify', 'tools': ['perceive', 'triage', 'finish']},
        {'id': 'report', 'tools': ['perceive', 'triage', 'draft', 'request_review']},
        {'id': 'skip_evidence', 'tools': ['triage', 'draft', 'request_review']},
    ]

In [ ]:
def verify_plan(intent, plan_id):
    # Change a rule here, then rerun training; this function is sent to Gym unchanged.
    expected = {'describe': ['perceive', 'finish'], 'triage': ['perceive', 'triage', 'finish'],
                'draft': ['perceive', 'triage', 'draft', 'request_review']}
    if intent not in expected:
        raise ValueError('Unknown task intent')
    plans = {p['id']: p['tools'] for p in plan_library()}
    if plan_id not in plans:
        return {'reward': 0.0, 'transitions': [], 'failures': ['Unknown plan']}
    observed = classified = drafted = False
    transitions, failures = [], []
    for tool in plans[plan_id]:
        allowed = ((tool == 'perceive' and not observed)
                   or (tool == 'triage' and observed and not classified)
                   or (tool == 'draft' and classified and not drafted)
                   or (tool == 'request_review' and drafted)
                   or (tool == 'finish' and observed))
        transitions.append({'tool': tool, 'allowed': allowed})
        if not allowed:
            failures.append('Evidence dependency failed: ' + tool)
            break
        observed |= tool == 'perceive'
        classified |= tool == 'triage'
        drafted |= tool == 'draft'
    if plans[plan_id] != expected[intent]:
        failures.append('Plan does not match task intent')
    return {'reward': float(not failures), 'transitions': transitions, 'failures': failures}

assert verify_plan('draft', 'skip_evidence')['reward'] == 0
assert verify_plan('draft', 'report')['reward'] == 1

### Define and run the optimizer

Gym dependencies remain in their own environment. The function below is the actual experiment; the launcher copies this cell’s function source verbatim. One update per fresh group means the importance ratio starts at one and clipping is inactive. The hash-checked NVIDIA loss kernels remain imported.

In [ ]:
def train_policy(output, steps=40, group_size=16, learning_rate=0.12, seed=42):
    import json, hashlib, sys, secrets
    from pathlib import Path
    from typing import ClassVar
    import torch
    from fastapi.testclient import TestClient
    from nemo_gym.base_resources_server import (
        BaseResourcesServerConfig, BaseVerifyRequest, BaseVerifyResponse, SimpleResourcesServer, ReverifyMode,
    )
    from nemo_gym.config_types import BaseServerConfig
    from nemo_gym.server_utils import ServerClient
    from omegaconf import OmegaConf
    from pydantic import PrivateAttr
    root = Path.cwd()  # run_in_gym launches from the repository root.
    sys.path.insert(0, str(root / 'run/workshop/gym'))
    from rl_kernel import load_kernels  # Hash-checked NVIDIA loss functions; no training loop here.

    class Config(BaseResourcesServerConfig):
        REVERIFY_MODE: ClassVar[ReverifyMode] = ReverifyMode.STATELESS

    class Verification(BaseVerifyResponse):
        transitions: list[dict]
        failures: list[str]

    class Resources(SimpleResourcesServer):
        config: Config
        _key: str = PrivateAttr(default_factory=lambda: secrets.token_urlsafe(32))

        def get_session_middleware_key(self):
            return self._key

        async def verify(self, body: BaseVerifyRequest) -> Verification:
            # The harness supplies intent; the policy can only select a plan ID.
            intent = body.responses_create_params.input
            try:
                action = json.loads(body.response.output[0].content[0].text)
                if set(action) != {'plan_id'} or not isinstance(action['plan_id'], str):
                    raise ValueError('Expected only a plan_id string')
                result = verify_plan(intent, action['plan_id'])
            except (ValueError, KeyError, IndexError, TypeError, AttributeError):
                result = {'reward': 0.0, 'transitions': [], 'failures': ['Invalid policy action']}
            return Verification(**body.model_dump(), **result)

    config = Config(host='127.0.0.1', port=7862, entrypoint='notebook-experiment.py', name='notebook_plans')
    server_client = ServerClient(head_server_config=BaseServerConfig(host='127.0.0.1', port=7862),
                                 global_config_dict=OmegaConf.create({}))
    app = Resources(config=config, server_client=server_client).setup_webserver()
    plans, intents = plan_library(), ['describe', 'triage', 'draft']
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    logits = torch.nn.Parameter(torch.zeros(len(intents), len(plans)))
    optimizer = torch.optim.Adam([logits], lr=learning_rate)
    kernels, manifest = load_kernels()
    history, rollouts = [], []
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)

    with TestClient(app) as client:
        def verify(intent, action):
            payload = {'responses_create_params': {'input': intent}, 'response': {
                'id': 'offline-plan', 'created_at': 0, 'model': 'tiny-policy', 'object': 'response',
                'parallel_tool_calls': False, 'tool_choice': 'none', 'tools': [],
                'output': [{'id': 'action', 'type': 'message', 'role': 'assistant', 'status': 'completed',
                            'content': [{'type': 'output_text', 'annotations': [],
                                         'text': json.dumps({'plan_id': plans[action]['id']})}]}]}}
            response = client.post('/verify', json=payload)
            response.raise_for_status()
            result = response.json()
            if result.get('failure_kind') or result.get('mask_sample') or result.get('failure_reason'):
                raise RuntimeError('Gym infrastructure failure: ' + str(result))
            return result

        reward_matrix = torch.tensor([[verify(intent, i)['reward'] for i in range(len(plans))] for intent in intents])
        def snapshot():
            probabilities = logits.detach().softmax(-1)
            weights = logits.detach().tolist()
            return {'probabilities': probabilities.tolist(), 'logits': weights,
                    'expected_reward': float((probabilities * reward_matrix).sum(-1).mean()),
                    'sha256': hashlib.sha256(json.dumps(weights).encode()).hexdigest()}
        before = snapshot()
        for step in range(steps):
            actions = torch.distributions.Categorical(logits=logits.detach()).sample((group_size,)).T
            old_logprobs = logits.detach().log_softmax(-1).gather(1, actions).reshape(-1, 1)
            samples = []
            for task_index, intent in enumerate(intents):
                for action in actions[task_index].tolist():
                    result = verify(intent, action)
                    samples.append({'step': step, 'intent': intent, 'plan_id': plans[action]['id'],
                                    'reward': result['reward'], 'transitions': result['transitions'],
                                    'failures': result['failures']})
            rewards = torch.tensor([row['reward'] for row in samples])
            prompt_ids = torch.arange(len(intents)).repeat_interleave(group_size).unsqueeze(-1)
            baseline, std, trivial = kernels['calculate_baseline_and_std_per_prompt'](
                prompt_ids, rewards, torch.ones_like(rewards), leave_one_out_baseline=False)
            advantages = torch.where(trivial, 0, (rewards-baseline)/std.clamp_min(1e-8)).unsqueeze(-1)
            current = logits.log_softmax(-1).gather(1, actions).reshape(-1, 1)
            mask = torch.ones_like(current)
            loss, _, _ = kernels['clipped_pg_actor_objective'](
                current, prev_logprobs=old_logprobs, advantages=advantages,
                token_mask=mask, mask=mask, sample_mask=mask[:, 0], importance_weights=None,
                global_valid_toks=mask.sum(), global_valid_seqs=mask.sum(),
                ratio_clip_min=0.2, ratio_clip_max=0.2, ratio_clip_c=0.0,
                disable_ppo_ratio=False, force_on_policy_ratio=False,
                sequence_level_importance_ratios=False, use_cispo=False, token_level_loss=True)
            optimizer.zero_grad()
            loss.backward()
            assert torch.isfinite(logits.grad).all()
            optimizer.step()
            history.append({'step': step, 'loss': loss.item(), 'expected_reward': snapshot()['expected_reward']})
            for i, row in enumerate(samples):
                row['advantage'] = float(advantages[i, 0])
                row['old_log_probability'] = float(old_logprobs[i, 0])
            rollouts.extend(samples)
        after = snapshot()
        draft_index = intents.index('draft')
        probabilities = after['probabilities'][draft_index]
        chosen = max(range(len(plans)), key=probabilities.__getitem__)
        bad = next(i for i, p in enumerate(plans) if p['id'] == 'skip_evidence')
        good = next(i for i, p in enumerate(plans) if p['id'] == 'report')
        rejected, replay = verify('draft', bad), verify('draft', chosen)
    comparison = {'failed_plan_reward': rejected['reward'], 'failed_plan_reason': rejected['failures'],
                  'before_correct_plan_probability': before['probabilities'][draft_index][good],
                  'after_correct_plan_probability': probabilities[good],
                  'before_skip_evidence_probability': before['probabilities'][draft_index][bad],
                  'after_skip_evidence_probability': probabilities[bad],
                  'before_expected_reward': before['expected_reward'], 'after_expected_reward': after['expected_reward'],
                  'replayed_plan': plans[chosen]['id'], 'replay_reward': replay['reward'],
                  'replay_transitions': replay['transitions'], 'optimizer_steps': steps, 'fresh_rollouts': len(rollouts),
                  'weights_changed': before['sha256'] != after['sha256'], 'lightning_weights_changed': False}
    (output / 'comparison.json').write_text(json.dumps(comparison, indent=2))
    (output / 'checkpoint.json').write_text(json.dumps(after, indent=2))
    (output / 'demo.json').write_text(json.dumps({'kind': 'editable notebook CPU experiment',
        'before': before, 'after': after, 'steps': history, 'rl_commit': manifest['commit'],
        'parameters': logits.numel(), 'evaluation': 'same finite training environment'}, indent=2))
    (output / 'rollouts.jsonl').write_text(''.join(json.dumps(row)+'\n' for row in rollouts))
    print(json.dumps(comparison, indent=2))

In [ ]:
from course.agent.notebook_support import run_in_gym
# The transport writes the exact functions above to notebook-experiment.py and runs them.
# Change these parameters or verify_plan above, then rerun this cell.
run_in_gym([plan_library, verify_plan, train_policy], 'train_policy', gym_output,
           steps=40, group_size=16, learning_rate=0.12, seed=42)
learning_result = json.loads((gym_output / 'comparison.json').read_text())

### Run the separate invalid-classification regression

This uses the student agent exported by notebook 09 with a scripted model response; it is independent of policy training.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('student_regression', OUTPUT / 'student-agent.py')
student_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(student_agent)
student_agent.SKILLS = OUTPUT / 'student-skills'
actions = iter(['perceive', 'triage', 'request_review', 'finish'])
def regression_model(model, messages, **kwargs):
    if model == 'specialist': return json.dumps(case['injected_specialist'])
    return json.dumps({'tool': next(actions), 'reason': 'Invalid classification must hold'})
regression_run = student_agent.run(model_call=regression_model)
assert regression_run['status'] == 'HOLD' and 'draft' not in regression_run['state']
regression_result = {'test_type': 'injected regression', 'regression_outcome': regression_run['status'],
                     'draft_created': False, 'saved_run_contract_failures': check_trace(record)}
(OUTPUT / 'regression-result.json').write_text(json.dumps(regression_result, indent=2))
display(JSON(regression_result))

In [ ]:
learning = learning_result
table(["Measurement", "Actual result"], [("Fresh attempts", learning["fresh_rollouts"]), ("Optimizer steps", learning["optimizer_steps"]), ("Weights changed", learning["weights_changed"]), ("Rejected plan reward", learning["failed_plan_reward"])])

### 4 · Compare and replay

These probabilities describe the small training environment—not performance on unseen tasks. Inspect the replayed plan and its fresh Gym result.

In [ ]:
bars({"Correct plan before": learning["before_correct_plan_probability"], "Correct plan after": learning["after_correct_plan_probability"], "Skip evidence before": learning["before_skip_evidence_probability"], "Skip evidence after": learning["after_skip_evidence_probability"]}, maximum=1)
print("Chosen plan:", learning["replayed_plan"])
table(["Tool", "Allowed"], [(s["tool"],s["allowed"]) for s in learning["replay_transitions"]])
print("Fresh Gym reward:", learning["replay_reward"])

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
assert learning_result['weights_changed'] and learning_result['failed_plan_reward'] == 0
assert regression_result['regression_outcome'] == 'HOLD'
manual_result = lesson.finish({'gym_rl': learning_result, 'regression': regression_result})
display(JSON(manual_result, expanded=False))

### Optional · Set up the full Gym/RL exercise for Lightning

**Exercise goal:** improve the notebook 09 controller's evidence gathering and tool decisions on held-out tasks. Start with a separate trainable Lightning controller checkpoint and keep the specialist frozen. This isolates whether the controller improves. To improve the specialist itself after `train_lightning.py`, design a separate task whose generated specialist answers are graded against trusted labels; the plan-selection reward below is insufficient to train answer quality.

The following is an implementation walkthrough, not a ready-to-run full trainer bundled with this course. The CPU checkpoint contains twelve plan logits and cannot initialize Lightning. Neither these Markdown instructions nor Run All launches GPU RL.

#### 1 · Provision an independent training stack

Follow NVIDIA's [NeMo RL installation instructions](https://docs.nvidia.com/nemo/rl/nightly/about/installation.html) and [Lightning model guide](https://docs.nvidia.com/nemo/rl/nightly/guides/models/nemotron/nemotron-3.5-lightning.html). Choose and record a compatible NeMo RL commit, its Gym dependencies, model revision, tokenizer and supported training/generation backend. Reserve training and rollout GPU capacity; use a separate output/checkpoint directory. The workshop's pinned Gym 0.6.0 environment and selected loss kernels are not a full NeMo RL installation. Nightly documentation evolves, so use the instructions for the selected revision rather than mixing its dependencies into the course environment.

#### 2 · Turn the failure into tasks and trusted rewards

Create disjoint training and held-out task files, split by scenario/clip rather than near-duplicate prompts. Each task needs an ID, user request, evidence fixture, allowed tools and private expected outcome. Include ordinary successful cases as well as the notebook's failure case. Keep expected answers in the verifier, outside model-visible prompts.

For this exercise, define reward 1 only when the required evidence was actually obtained, the classification passes the trusted schema/label checks, and the requested final outcome is correct; otherwise reward 0. Include missing evidence, invalid classification, unsupported claims and omitted review as negative cases. Verify the known bad trace scores 0 and a valid reference trajectory scores 1 before training. This is a proposed workshop reward design, not an NVIDIA-provided benchmark.

#### 3 · Implement the Gym environment

Use NVIDIA's [Gym Agent server lifecycle](https://docs.nvidia.com/nemo/gym/v0.6.0/agent-server/) and [environment tutorials](https://docs.nvidia.com/nemo/gym/v0.6.0/environment-tutorials/) as interface references; adapt them to the Gym version selected with the trainer.

Wrap notebook 09's agent loop in an Agent server. Reset episode state for each task, send model calls through Gym's Model server, route tools through a Resources server, and finish by calling the trusted verifier. Put evidence/tool state and reward computation in Resources. Set turn limits and timeouts, and use isolated fixtures for any action with side effects. Do not let a model-written plan or self-reported success stand in for tool execution evidence.

#### 4 · Connect rollouts to the full trainer

Start from NVIDIA's `examples/nemo_gym/run_grpo_nemo_gym.py` and its associated configurations in the **selected NeMo RL checkout**. The [official integration guide](https://docs.nvidia.com/nemo/rl/nightly/design-docs/nemo-gym-integration.html) documents this partial configuration; it is not a complete launch configuration:

```yaml
policy:
  generation:
    backend: vllm
    vllm_cfg:
      async_engine: true
      expose_http_server: true
env:
  should_use_nemo_gym: true
  nemo_gym:
    config_paths:
      - resources_servers/workshop/configs/workshop.yaml
      - responses_api_agents/workshop/configs/workshop.yaml
```

The two `workshop.yaml` files are paths you must implement in the Gym workspace; they are not supplied by this repository. Add the model, task datasets, resource allocation, generation limits, optimization settings and checkpoint configuration using that revision's complete example. Keep Python and Ray versions aligned across RL and Gym.

Use trainer-managed generation with weight synchronization. Collect fresh conversations, generated token IDs, sampled log probabilities, training masks, tool responses and verifier rewards. Relay traces identify failures but are not sufficient on-policy training tensors. Gym orchestrates and verifies on CPU; the full model's training and generation consume GPU resources. The existing fixed inference endpoint is not automatically a trainable rollout worker.

#### 5 · Run a small experiment, then measure improvement

First run evaluation-only rollouts to check tool execution, reward correctness and termination. Record a frozen baseline on held-out tasks. Then launch a short GRPO job using the selected upstream example's launch instructions, with a small explicit step/sample budget. Inspect nonempty generated tokens, finite loss, reward variation, policy updates and checkpoint creation. If every sample receives the same reward, inspect tasks and the verifier before extending training.

Evaluate the trained controller on the same held-out tasks with the same frozen specialist and decoding settings. Compare task success, invalid classifications, unsupported conclusions, review compliance and tool-call cost; inspect failed trajectories and reward exploitation. Save the full configuration, revisions, seed, dataset hashes, checkpoints and evaluation report. Promote a separate serving candidate only after these checks demonstrate improvement. Adapter serving must satisfy notebook 05's model/backend compatibility requirements.

**Deliverables:** task splits, an Agent server, a Resources verifier with positive/negative cases, a complete trainer configuration, baseline rollouts, a trained model checkpoint and held-out comparison. The executable CPU exercise in this notebook supplies an inspectable learning example; these deliverables are the additional work needed for full Lightning RL.

### Grow the skill library
The library now includes this lesson’s skill, **workshop-review-trace**. Open the actual instructions below.

**`inspect-trace` → `test-regression` → `check-regression`**

In [ ]:
show_skill_library('11-trace-and-improve')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-review-trace/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-review-trace/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `08-agent-skills`, `09-agent-loop`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('11-trace-and-improve')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-review-trace/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-trace', 'test-regression', 'check-regression']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano coordinates the lesson tools. Relay supplies failure evidence, Gym scores fresh plans, and NeMo RL loss functions update the tiny policy. Lightning and its adapter remain unchanged.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** Missing evidence → reward 0; fresh RL updates → compare plan probabilities and replay.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** Use traces to identify the problem, a verifier to score behavior, and fresh tests to measure a change. Choose between skill, tool, policy, prompt, or model changes based on the evidence.

Exit the TUI with `/exit` when done.